<a href="https://colab.research.google.com/github/checheanya/liver_cell_gnn/blob/dev_executution/my_runs/liver_cell_gnn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Liver cell GNN — Colab workspace

Use this notebook to:
1. Install dependencies
2. Set **your** data and repo paths
3. Run a **mock** sanity check (no real WSI required)

**Tip:** In Colab, use *Runtime → Change runtime type → GPU* for CUDA tests.

## 1) Environment installation

Run the cell below once per new runtime. Versions are pinned loosely for reproducibility; adjust if something conflicts with Colab’s default PyTorch.

In [1]:
# check CUDA version
!nvcc --version
# python version
!python --version
# check PyTorch version
!pip show torch

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Python 3.12.13
Name: torch
Version: 2.4.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-nccl-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, deepsnap, dgl, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision


In [2]:
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Core stack (add/remove as needed for your repo)
pip_install([
    "yacs",
    "lifelines",
    "scikit-learn",
    "pandas",
    "tqdm",
    "networkx",
    "Pillow"
])

# DGL: pick ONE line that matches your environment (see https://www.dgl.ai/pages/start.html)
# Trying CUDA 12.2 wheel for compatibility with Colab's CUDA 12.8 and PyTorch 2.10.0+cu128
#pip_install(["dgl", "-f", "https://data.dgl.ai/wheels/cu122/repo.html"])
# CPU wheel (safe default when CUDA match is unclear):
#pip_install(["dgl", "-f", "https://data.dgl.ai/wheels/repo.html"])

print("pip installs finished.")

pip installs finished.


In [3]:
!pip uninstall dgl

Found existing installation: dgl 2.4.0+cu124
Uninstalling dgl-2.4.0+cu124:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/dgl-2.4.0+cu124.dist-info/*
    /usr/local/lib/python3.12/dist-packages/dgl/*
Proceed (Y/n)? Y
  Successfully uninstalled dgl-2.4.0+cu124


In [2]:
!!pip install --pre --upgrade --no-cache-dir torch --extra-index-url https://download.pytorch.org/whl/nightly/cu128
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!pip install torchdata
# PyTorch Geometric & DeepSNAP — only if you import GraphLab loader end-to-end
# Uncomment when you need full training pipeline:
!pip install -q torch-geometric
!pip install -q deepsnap

# check DGL version
!pip show dgl

Looking in links: https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
Name: dgl
Version: 2.4.0+cu124
Summary: Deep Graph Library
Home-page: https://github.com/dmlc/dgl
Author: 
Author-email: 
License: APACHE
Location: /usr/local/lib/python3.12/dist-packages
Requires: networkx, numpy, packaging, pandas, psutil, pydantic, pyyaml, requests, scipy, torch, tqdm
Required-by: 


In [9]:
!pwd
!git clone https://github.com/checheanya/liver_cell_gnn.git

/content
Cloning into 'liver_cell_gnn'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 140 (delta 35), reused 67 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 110.68 KiB | 5.27 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [10]:
!mkdir sample_lihc
!mv TCGA* sample_lihc/

## 2) Google Drive (optional)

If your data and/or `liver_cell_gnn` live on Drive, mount and use paths like `/content/drive/MyDrive/...`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab — skip Drive mount.")

## 3) Paths — **fill in your values**

| Variable | Purpose |
|----------|--------|
| `REPO_ROOT` | Folder that contains the `GraphLab/` and `Run/` packages (parent of `liver_cell_gnn` if you keep that name) |
| `DATA_ROOT` | Root for datasets (graphs, CSVs, etc.) |
| `OUTPUT_ROOT` | Logs, checkpoints, notebook outputs |

Example (Colab + Drive): `REPO_ROOT = "/content/drive/MyDrive/projects/liver_cell_gnn"`

In [5]:
from pathlib import Path

# --- EDIT THESE ---
REPO_ROOT = Path("/content/liver_cell_gnn")  # contains GraphLab/, Run/
DATA_ROOT = Path("/content/sample_lihc")
OUTPUT_ROOT = Path("/content/results")
# ------------------

for p in (DATA_ROOT, OUTPUT_ROOT):
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())

REPO_ROOT: /content/liver_cell_gnn
DATA_ROOT: /content/sample_lihc
OUTPUT_ROOT: /content/results


In [6]:
import os
import sys

# So `import GraphLab` works when REPO_ROOT points at the repo root (same layout as local `Run/main.py`)
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))
    print("Added to sys.path:", REPO_ROOT)
else:
    print("WARNING: REPO_ROOT does not exist yet — clone/upload the repo or fix the path.")

Added to sys.path: /content/liver_cell_gnn


## 4) Mock test run

Lightweight checks: **PyTorch**, **DGL**, optional **GraphLab** import. No WSI or `AllCell.bin` required.

If GraphLab fails to import, install missing deps from the error message or uncomment PyG/DeepSNAP above.

In [1]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

x = torch.randn(4, 8, device="cuda" if torch.cuda.is_available() else "cpu")
y = x @ x.T
print("matmul shape:", y.shape, "ok")

torch: 2.12.0.dev20260325+cu128
cuda available: True
device: NVIDIA RTX PRO 6000 Blackwell Server Edition
matmul shape: torch.Size([4, 4]) ok


In [2]:
import dgl
import torch as th

g = dgl.graph(([0, 0, 1], [1, 2, 2]), num_nodes=3)
g.ndata["feat"] = th.randn(3, 16)
print("DGL graph:", g)
print("nodes:", g.num_nodes(), "edges:", g.num_edges())

DGL graph: Graph(num_nodes=3, num_edges=3,
      ndata_schemes={'feat': Scheme(shape=(16,), dtype=torch.float32)}
      edata_schemes={})
nodes: 3 edges: 3


In [ ]:
# Optional: import GraphLab (needs REPO_ROOT on sys.path and full dependency tree)
try:
    import GraphLab
    from GraphLab.config import cfg
    print("GraphLab import OK, cfg.seed =", cfg.seed)
except Exception as e:
    print("GraphLab import skipped or failed:", type(e).__name__, e)
    print("Install missing packages from the message, or run training only on a machine with the full env.")

In [ ]:
# Write a tiny marker file to OUTPUT_ROOT to confirm paths are writable
marker = OUTPUT_ROOT / "mock_run_ok.txt"
marker.write_text("mock run completed\n")
print("Wrote:", marker)

### Next steps

- Copy your **`Run/configs/IDGNN/graph.yaml`** and set `dataset.dir`, `out_dir`, etc. to paths under `DATA_ROOT` / `OUTPUT_ROOT`.
- From a terminal or `!cd` cell, run: `python main.py --cfg configs/IDGNN/graph.yaml` inside `Run/` once the full dataset is in place.
- See `liver_cell_gnn/README.md` → **Getting Started** for the on-disk layout (`train/val/test`, `AllCell.bin`).